## Очистка датасета от отрисованных линий

In [1]:
import cv2
import numpy as np
import json
import shutil
from pathlib import Path
from tqdm import tqdm
import time
from collections import Counter

ROOT_DATASET_DIR = Path("C:/Recognition of unpaved roads")
OUTPUT_ROOT_DIR = Path("Recognition of unpaved roads")

LINE_THICKNESS = 3
INPAINT_RADIUS = 1
DILATE_SIZE = 2


def create_polyline_mask(image_shape, polylines, thickness=2):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    for polyline in polylines:
        pts = np.array(polyline, dtype=np.int32)
        cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=thickness)
    return mask


def remove_lines(image, mask, radius=1, dilate=2):
    if np.sum(mask) < 100:
        return image
    
    if dilate > 0:
        kernel = np.ones((dilate*2+1, dilate*2+1), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    
    result = cv2.inpaint(image, mask, inpaintRadius=radius, flags=cv2.INPAINT_TELEA)
    return result


def process_single_image(image_path, json_path, output_image_path, output_json_path):
    try:
        image = cv2.imread(str(image_path))
        if image is None:
            return False
        
        with open(json_path, 'r', encoding='utf-8') as f:
            annotation = json.load(f)
        
        polylines = annotation.get('polylines', [])
        if len(polylines) == 0:
            cv2.imwrite(str(output_image_path), image)
            shutil.copy2(json_path, output_json_path)
            return True
        
        mask = create_polyline_mask(image.shape, polylines, LINE_THICKNESS)
        image_clean = remove_lines(image, mask, INPAINT_RADIUS, DILATE_SIZE)
        
        output_image_path.parent.mkdir(parents=True, exist_ok=True)
        output_json_path.parent.mkdir(parents=True, exist_ok=True)
        
        cv2.imwrite(str(output_image_path), image_clean)
        shutil.copy2(json_path, output_json_path)
        
        return True
        
    except Exception as e:
        print(f"Error: {image_path.name} - {e}")
        return False


def find_pairs_universal(root_dir):
    pairs = []
    
    direct_images = root_dir / "images"
    direct_annotations = root_dir / "annotations"
    
    if direct_images.exists() and direct_annotations.exists():
        for json_path in direct_annotations.glob("*.json"):
            image_id = json_path.stem
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                img_path = direct_images / f"{image_id}{ext}"
                if img_path.exists():
                    pairs.append({
                        'image_path': img_path,
                        'json_path': json_path,
                        'relative_path': "",
                        'subdir_name': root_dir.name
                    })
                    break
    
    for subdir in root_dir.iterdir():
        if subdir.is_dir():
            images_dir = subdir / "images"
            annotations_dir = subdir / "annotations"
            
            if images_dir.exists() and annotations_dir.exists():
                for json_path in annotations_dir.glob("*.json"):
                    image_id = json_path.stem
                    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                        img_path = images_dir / f"{image_id}{ext}"
                        if img_path.exists():
                            pairs.append({
                                'image_path': img_path,
                                'json_path': json_path,
                                'relative_path': subdir.name,
                                'subdir_name': subdir.name
                            })
                            break
    
    return pairs


def clean_dataset():
    OUTPUT_ROOT_DIR.mkdir(parents=True, exist_ok=True)
    
    all_pairs = find_pairs_universal(ROOT_DATASET_DIR)
    
    print("="*60)
    print("DATASET CLEANING - POLYLINES")
    print("="*60)
    print(f"Input: {ROOT_DATASET_DIR}")
    print(f"Output: {OUTPUT_ROOT_DIR}")
    print(f"Total pairs found: {len(all_pairs)}")
    print(f"Line thickness: {LINE_THICKNESS}")
    print(f"Method: Telea, radius={INPAINT_RADIUS}, dilate={DILATE_SIZE}")
    print("="*60)
    
    if len(all_pairs) == 0:
        print("No pairs found. Check folder structure.")
        return
    
    folder_counts = Counter([p['subdir_name'] for p in all_pairs])
    print("\nDistribution:")
    for folder, count in folder_counts.items():
        if folder == ROOT_DATASET_DIR.name:
            print(f"  root: {count} images")
        else:
            print(f"  {folder}: {count} images")
    
    print("\nProcessing...\n")
    
    success = 0
    start = time.time()
    
    for pair in tqdm(all_pairs, desc="Progress"):
        if pair['relative_path'] == "":
            output_images_dir = OUTPUT_ROOT_DIR / "images"
            output_json_dir = OUTPUT_ROOT_DIR / "annotations"
        else:
            output_images_dir = OUTPUT_ROOT_DIR / pair['relative_path'] / "images"
            output_json_dir = OUTPUT_ROOT_DIR / pair['relative_path'] / "annotations"
        
        out_img = output_images_dir / pair['image_path'].name
        out_json = output_json_dir / pair['json_path'].name
        
        if process_single_image(pair['image_path'], pair['json_path'], out_img, out_json):
            success += 1
    
    elapsed = time.time() - start
    
    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)
    print(f"Successful: {success}/{len(all_pairs)}")
    print(f"Errors: {len(all_pairs) - success}")
    print(f"Time: {elapsed:.1f} sec")
    print(f"Speed: {success/elapsed:.2f} images/sec")
    print(f"Output: {OUTPUT_ROOT_DIR}")
    print("="*60)
    
    report = {
        "processing_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "input_root": str(ROOT_DATASET_DIR),
        "output_root": str(OUTPUT_ROOT_DIR),
        "total_images": len(all_pairs),
        "successful": success,
        "errors": len(all_pairs) - success,
        "processing_time_seconds": elapsed,
        "params": {
            "line_thickness": LINE_THICKNESS,
            "inpaint_radius": INPAINT_RADIUS,
            "dilate_size": DILATE_SIZE,
            "method": "Telea"
        },
        "folders": dict(folder_counts)
    }
    
    report_path = OUTPUT_ROOT_DIR / "cleaning_report.json"
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    
    print(f"\nReport saved: {report_path}")


if __name__ == "__main__":
    clean_dataset()

DATASET CLEANING - POLYLINES
Input: C:\Recognition of unpaved roads
Output: Recognition of unpaved roads
Total pairs found: 1345
Line thickness: 3
Method: Telea, radius=1, dilate=2

Distribution:
  1_stage: 336 images
  2_stage: 337 images
  3_stage: 337 images
  4_stage: 335 images

Processing...



Progress: 100%|██████████| 1345/1345 [07:30<00:00,  2.99it/s]


RESULTS
Successful: 1345/1345
Errors: 0
Time: 450.1 sec
Speed: 2.99 images/sec
Output: Recognition of unpaved roads

Report saved: Recognition of unpaved roads\cleaning_report.json
